# TPU lane-padding: `(N, 1)` vs `(N,)` ket storage

On a TPU the minor (last) array axis is padded up to the **128**-element lane width (and the second-minor axis to the 8-element sublane width). A ket stored as an `(N, 1)` column therefore pads its size-1 minor axis to 128 — the physical buffer becomes **~128×** larger than the `N` numbers it holds.

jaxquantum now stores kets/bras as 1-D `(N,)` arrays, so the minor axis is the Hilbert space itself and no longer pads to a singleton. This notebook **measures** the difference so you can confirm the win on real TPU hardware.

> On CPU/GPU there is no lane padding, so the ratios below are ~1. Run this on a TPU host to see the blow-up.

See also the headless script `measure_ket_padding.py` in this folder.

In [ ]:
import jax
import jax.numpy as jnp
import jaxquantum as jqt

platform = jax.devices()[0].platform
print('JAX backend platform:', platform)
if platform != 'tpu':
    print('NOTE: not on a TPU — ratios will be ~1. Run on a TPU host to see (N,1)->(N,128) padding.')

In [ ]:
def output_bytes(x):
    """Physical (padded) output-buffer size in bytes of an identity copy of x."""
    compiled = jax.jit(lambda a: a + 0).lower(x).compile()
    return int(compiled.memory_analysis().output_size_in_bytes)

def hlo_layout(x):
    text = jax.jit(lambda a: a + 0).lower(x).compile().as_text()
    for line in text.splitlines():
        if 'parameter(0)' in line:
            return line.strip()
    return text.splitlines()[0].strip()

In [ ]:
print(f"{'shape (old)':>14} {'shape (new)':>12} {'old bytes':>12} {'new bytes':>12} {'ratio':>7}")
print('-' * 64)
for N, B in [(128, None), (1024, None), (4096, None), (1024, 8)]:
    if B is None:
        old = jnp.ones((N, 1), dtype=jnp.complex128); new = jnp.ones((N,), dtype=jnp.complex128)
        os_, ns_ = f'({N}, 1)', f'({N},)'
    else:
        old = jnp.ones((B, N, 1), dtype=jnp.complex128); new = jnp.ones((B, N), dtype=jnp.complex128)
        os_, ns_ = f'({B}, {N}, 1)', f'({B}, {N})'
    ob, nb = output_bytes(old), output_bytes(new)
    print(f'{os_:>14} {ns_:>12} {ob:>12,} {nb:>12,} {ob/max(nb,1):>6.1f}x')

## HLO layout

The compiled HLO shows the buffer shape/layout. On a TPU the old `(N, 1)` becomes `c128[N,128]{...}` while the new `(N,)` stays `c128[N]{0}`.

In [ ]:
print('old (N,1):', hlo_layout(jnp.ones((4096, 1), dtype=jnp.complex128)))
print('new (N,) :', hlo_layout(jnp.ones((4096,), dtype=jnp.complex128)))

## End-to-end: the `sesolve` carry stays 1-D

`jqt.basis` now returns a 1-D ket, and the Schrödinger solver keeps the state 1-D through the integration (no `(N,1)` is reintroduced in the `lax.scan`/diffrax carry).

In [ ]:
N = 1024
psi = jqt.basis(N, 1)
print('jqt.basis(N,1).data.shape =', psi.data.shape, '(expected (N,))')
print('carry buffer layout       :', hlo_layout(psi.data), '(expect [N], not [N,1]/[N,128])')

# Run a short evolution and confirm the saved states are 1-D per time step.
H = jqt.create(N) @ jqt.destroy(N)
ts = jnp.linspace(0.0, 1.0, 5)
states = jqt.sesolve(H, psi, ts)
print('sesolve states.data.shape =', states.data.shape, '(expected (T, N))')